# Pandas from First Principles
## Notebook 11: Reading & Writing Data

---

> **Series Overview:** This notebook is part of a beginner-to-intermediate Pandas series.  
> **Prerequisites:** Notebooks 1–10 (DataFrames, indexing, groupby, merging, etc.)  
> **What you'll learn:** How to read data from CSV, Excel, and JSON — and write it back out cleanly.

---

## 1. Why I/O Matters

In every real project, data doesn't live inside your Python script — it arrives from **external sources**:

- CSV files exported from databases or Excel
- Excel spreadsheets shared by colleagues
- JSON responses from web APIs
- Database query results dumped to files

### The General Pattern

```
Read  →  Clean  →  Analyze  →  Write
```

Understanding **how to read data correctly** (right types, right encoding, handling missing values) saves you hours of debugging later.

### Most Common Formats

| Format | Pandas Read      | Pandas Write     | Notes                        |
|--------|------------------|------------------|------------------------------|
| CSV    | `pd.read_csv()`  | `df.to_csv()`    | Most universal format        |
| Excel  | `pd.read_excel()`| `df.to_excel()`  | Requires `openpyxl`          |
| JSON   | `pd.read_json()` | `df.to_json()`   | Common for APIs              |

Let's start with the most important one: **CSV**.

In [ ]:
# Standard imports used throughout this notebook
import pandas as pd
import io  # io.StringIO lets us simulate a file from a string — no disk file needed!

print("pandas version:", pd.__version__)
print("io module loaded — we can simulate CSV/JSON files in memory.")

---
## 2. `pd.read_csv()` — The Most Used Pandas Function

### What is it?
`pd.read_csv()` reads a CSV (Comma-Separated Values) file into a DataFrame.

### Why use it?
CSV is the **most common data exchange format** in the world — databases export to it, Excel can save as it, and almost every data tool understands it.

### Full Syntax
```python
pd.read_csv(
    filepath_or_buffer,   # path to file, or a file-like object (e.g. io.StringIO)
    sep=',',              # delimiter between fields (default: comma)
    header=0,             # row number to use as column names (default: row 0)
    index_col=None,       # column to use as row index
    usecols=None,         # list of columns to read (saves memory)
    dtype=None,           # dict of column → type to enforce on load
    nrows=None,           # read only the first N rows
    skiprows=None,        # rows to skip from the top
    na_values=None        # extra strings to recognise as NaN
)
```

### Return Value
Returns a **DataFrame**.

---

### 2.1 Basic Reading with `io.StringIO`

> **`io.StringIO`** wraps a plain Python string so it behaves like a file object.  
> This lets us practice `read_csv()` without needing any actual file on disk.

In [ ]:
# --- 2.1 Basic read_csv() ---
# We define the CSV content as a multi-line Python string,
# then wrap it in io.StringIO so pandas treats it like a file.

csv_data = """name,age,salary,city
Alice,28,85000,Delhi
Bob,34,,Mumbai
Carol,26,91000,Pune
David,40,72000,Delhi
Eva,31,95000,Bangalore
"""

df = pd.read_csv(io.StringIO(csv_data))

print("=== DataFrame ===")
print(df)
print()
print("=== dtypes ===")
print(df.dtypes)
print()
print("=== Shape ===")
print(df.shape)

---
### 2.2 `sep` — Delimiter Parameter

**What:** Specifies which character separates fields in your file.  
**Default:** `','` (comma)  
**Why:** Not all CSVs use commas — European data often uses semicolons (`;`), and TSV files use tabs (`\t`).

```python
# Tab-separated:
pd.read_csv('data.tsv', sep='\t')

# Semicolon-separated:
pd.read_csv('data.csv', sep=';')
```

> **Common Mistake:** Opening a semicolon-separated file without `sep=';'` — you get one giant column instead of many.

In [ ]:
# --- 2.2 sep parameter ---
# Simulating a semicolon-separated file

semicolon_data = """name;department;score
Alice;Engineering;92
Bob;Marketing;78
Carol;Engineering;88
David;HR;85
"""

# Without sep=';' — wrong!
df_wrong = pd.read_csv(io.StringIO(semicolon_data))  # uses comma by default
print("=== Without sep=';' (WRONG) ===")
print(df_wrong.head())
print(f"Columns: {df_wrong.columns.tolist()}")
print()

# With sep=';' — correct!
df_correct = pd.read_csv(io.StringIO(semicolon_data), sep=';')
print("=== With sep=';' (CORRECT) ===")
print(df_correct)
print(f"Columns: {df_correct.columns.tolist()}")

---
### 2.3 `header` — Which Row Contains Column Names

**What:** The row number (0-indexed) to use as column headers.  
**Default:** `header=0` (first row)  
**Use `header=None`** when your file has no header row — Pandas will auto-generate column numbers (0, 1, 2…).

> **Tip:** Use `header=1` if the first row is metadata/notes and the second row is the real header.

In [ ]:
# --- 2.3 header parameter ---

# File with NO header row
no_header_data = """Alice,28,85000,Delhi
Bob,34,70000,Mumbai
Carol,26,91000,Pune
"""

# Without header=None — Pandas treats first data row as header!
df_bad = pd.read_csv(io.StringIO(no_header_data))
print("=== Without header=None (WRONG — loses first row!) ===")
print(df_bad)
print()

# With header=None — auto-generates column numbers
df_good = pd.read_csv(io.StringIO(no_header_data), header=None)
print("=== With header=None ===")
print(df_good)
print()

# Provide your own names
df_named = pd.read_csv(
    io.StringIO(no_header_data),
    header=None,
    names=['name', 'age', 'salary', 'city']
)
print("=== With header=None + names= ===")
print(df_named)

---
### 2.4 `index_col` — Setting the Row Index on Load

**What:** Specifies which column to use as the DataFrame's row index.  
**Value:** Column name (string) or column position (int), or `False` to prevent index inference.  
**Why:** Avoids a separate `.set_index()` call after reading.

> **Tip:** If your CSV already has an index column (e.g. from a previous `to_csv()` with `index=True`), use `index_col=0` to restore it correctly.

In [ ]:
# --- 2.4 index_col parameter ---

csv_data = """employee_id,name,department,salary
E001,Alice,Engineering,85000
E002,Bob,Marketing,70000
E003,Carol,Engineering,91000
E004,David,HR,68000
"""

# Default: integer RangeIndex
df_default = pd.read_csv(io.StringIO(csv_data))
print("=== Default index (RangeIndex) ===")
print(df_default)
print()

# Use employee_id as index
df_indexed = pd.read_csv(io.StringIO(csv_data), index_col='employee_id')
print("=== index_col='employee_id' ===")
print(df_indexed)
print()
print("Index values:", df_indexed.index.tolist())

---
### 2.5 `usecols` — Read Only Specific Columns

**What:** A list of column names (or positions) to load.  
**Why:** Massive memory saving on large files — if you have 50 columns but only need 5, don't load all 50!

```python
# Load only 'name' and 'salary'
pd.read_csv('big_file.csv', usecols=['name', 'salary'])
```

> **Best Practice:** Always use `usecols=` in production pipelines when you know which columns you need.

In [ ]:
# --- 2.5 usecols parameter ---

csv_data = """name,age,salary,city,department,join_year
Alice,28,85000,Delhi,Engineering,2019
Bob,34,70000,Mumbai,Marketing,2015
Carol,26,91000,Pune,Engineering,2021
David,40,68000,Delhi,HR,2012
"""

# Load ALL columns (uses more memory)
df_all = pd.read_csv(io.StringIO(csv_data))
print("=== All columns ===")
print(df_all.columns.tolist())
print()

# Load ONLY name, salary, city
df_selected = pd.read_csv(io.StringIO(csv_data), usecols=['name', 'salary', 'city'])
print("=== usecols=['name', 'salary', 'city'] ===")
print(df_selected)
print()
print(f"Memory saving: loaded {len(df_selected.columns)} of {len(df_all.columns)} columns")

---
### 2.6 `dtype` — Enforce Column Types on Load

**What:** A dictionary mapping `{column_name: type}` to force types during reading.  
**Why:** Pandas guesses types by scanning values. If a column has missing values, Pandas might read an int column as `float64` or `object`. Specifying `dtype=` avoids surprises.

```python
pd.read_csv('data.csv', dtype={'salary': float, 'zip_code': str})
```

> **Best Practice:** Always use `dtype={'zip_code': str}` for codes, IDs, or phone numbers — they look like numbers but should stay as strings.

In [ ]:
# --- 2.6 dtype parameter ---

csv_data = """name,age,salary,zip_code
Alice,28,85000.50,110001
Bob,34,70000.00,400001
Carol,26,91000.75,411001
"""

# Without dtype — Pandas guesses
df_auto = pd.read_csv(io.StringIO(csv_data))
print("=== Auto-detected dtypes ===")
print(df_auto.dtypes)
print(f"zip_code type: {df_auto['zip_code'].dtype}")  # reads as int64 — wrong for ZIP!
print()

# With dtype — we control the types
df_typed = pd.read_csv(
    io.StringIO(csv_data),
    dtype={'age': int, 'salary': float, 'zip_code': str}
)
print("=== Enforced dtypes ===")
print(df_typed.dtypes)
print(f"zip_code type: {df_typed['zip_code'].dtype}")  # str — correct!
print()
print(df_typed)

---
### 2.7 `nrows` and `skiprows` — Controlling Which Rows to Load

#### `nrows`
**What:** Read only the first N rows.  
**Why:** Perfect for **previewing** a huge file before committing to loading all of it.

#### `skiprows`
**What:** Skip a number of rows from the top (or a list of specific row indices).  
**Why:** Some files have metadata, report titles, or blank rows before the actual data starts.

```python
pd.read_csv('big_file.csv', nrows=1000)           # first 1000 rows only
pd.read_csv('report.csv', skiprows=3)             # skip first 3 rows
pd.read_csv('data.csv', skiprows=[0, 2, 4])       # skip rows 0, 2, 4
```

In [ ]:
# --- 2.7 nrows and skiprows ---

large_csv = """name,score
Alice,90
Bob,85
Carol,92
David,78
Eva,95
Frank,88
Grace,91
"""

# nrows: read only first 3 data rows
df_nrows = pd.read_csv(io.StringIO(large_csv), nrows=3)
print("=== nrows=3 (first 3 rows only) ===")
print(df_nrows)
print()

# skiprows: file with metadata at the top
messy_csv = """Report: Employee Scores - Q1 2024
Generated by HR System
name,score
Alice,90
Bob,85
Carol,92
"""

# Skip the 2 metadata rows at the top
df_skipped = pd.read_csv(io.StringIO(messy_csv), skiprows=2)
print("=== skiprows=2 (skip metadata rows) ===")
print(df_skipped)

---
### 2.8 `na_values` — Custom Missing Value Strings

**What:** A list of strings (in addition to the defaults) that should be treated as `NaN`.  
**Why:** Real-world data has many ways of expressing "missing": `'N/A'`, `'--'`, `'none'`, `'null'`, `'?'`, `'NA'`, empty string, etc. Pandas only catches some by default.

**Pandas default NaN strings include:** `''`, `'#N/A'`, `'N/A'`, `'NA'`, `'NULL'`, `'NaN'`, `'n/a'`, `'nan'`, `'null'`

> **Best Practice:** Always add `na_values=['--', 'none', '?', 'missing']` to catch non-standard sentinels from legacy systems.

In [ ]:
# --- 2.8 na_values parameter ---

# Data with non-standard missing value markers
messy_csv = """name,age,salary,city
Alice,28,85000,Delhi
Bob,--,N/A,Mumbai
Carol,26,91000,none
David,40,?,Delhi
Eva,31,95000,Pune
"""

# Without na_values — '--', 'none', '?' stay as strings
df_raw = pd.read_csv(io.StringIO(messy_csv))
print("=== Without na_values ===")
print(df_raw)
print(f"\nage dtype: {df_raw['age'].dtype}")
print(f"salary dtype: {df_raw['salary'].dtype}")
print()

# With na_values — they become proper NaN
df_clean = pd.read_csv(
    io.StringIO(messy_csv),
    na_values=['--', 'none', '?', 'N/A', 'missing']
)
print("=== With na_values=['--', 'none', '?', 'N/A'] ===")
print(df_clean)
print()
print("NaN count per column:")
print(df_clean.isna().sum())

---
## 3. `df.to_csv()` — Writing DataFrames to CSV

### What is it?
Writes a DataFrame to a CSV file (or any file-like object).

### Syntax
```python
df.to_csv(
    path_or_buf,    # file path or None (returns string)
    sep=',',        # field delimiter
    index=True,     # write row index? Almost always False!
    columns=None,   # list of columns to write (None = all)
    na_rep='',      # string to use for NaN values
    encoding='utf-8'
)
```

### Return Value
- If `path_or_buf=None` → returns the CSV as a **string**
- Otherwise → writes to file, returns `None`

---

### The `index=False` Rule

> ⚠️ **Almost always use `index=False`.**  
> By default, Pandas writes the row index as an extra column. When you read that file back, you get an unwanted `Unnamed: 0` column.

```
With index=True (default):          With index=False:
,name,age                           name,age
0,Alice,28                          Alice,28
1,Bob,34                            Bob,34
```

In [ ]:
# --- 3. to_csv() and the index=False rule ---

df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'age': [28, 34, 26],
    'salary': [85000, 70000, 91000]
})

# --- With default index=True (the problem) ---
# path=None returns the CSV as a string instead of writing to disk
csv_with_index = df.to_csv(index=True)  # default
print("=== to_csv(index=True) — DEFAULT — BAD ===")
print(csv_with_index)

# Read it back — see the problem!
df_back_bad = pd.read_csv(io.StringIO(csv_with_index))
print("After reading back:")
print(df_back_bad.columns.tolist())  # Has 'Unnamed: 0'!
print()

# --- With index=False (correct) ---
csv_no_index = df.to_csv(index=False)
print("=== to_csv(index=False) — CORRECT ===")
print(csv_no_index)

# Read it back — clean!
df_back_good = pd.read_csv(io.StringIO(csv_no_index))
print("After reading back:")
print(df_back_good.columns.tolist())  # Clean columns

---
### 3.2 `na_rep` and `columns` Parameters

#### `na_rep`
**What:** The string to use when writing `NaN` values. Default is `''` (empty string).  
**Why:** Some downstream systems need `'NULL'`, `'N/A'`, or `'0'` instead of blank.

#### `columns`
**What:** A list of column names to include in the output.  
**Why:** Write only the columns relevant to the recipient — hide internal columns.

In [ ]:
# --- 3.2 na_rep and columns ---
import numpy as np

df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David'],
    'age': [28, 34, 26, np.nan],
    'salary': [85000.0, np.nan, 91000.0, 68000.0],
    'internal_id': ['E001', 'E002', 'E003', 'E004']  # internal — don't share
})

print("=== Original DataFrame ===")
print(df)
print()

# Write only name + salary (not internal_id), represent NaN as 'N/A'
csv_output = df.to_csv(
    index=False,
    columns=['name', 'salary'],
    na_rep='N/A'
)
print("=== to_csv(columns=['name','salary'], na_rep='N/A') ===")
print(csv_output)

---
## 4. `pd.read_excel()` and `df.to_excel()`

### What is it?
Reads/writes Excel files (`.xlsx`, `.xls`).

### Installation
Excel I/O requires the `openpyxl` library:
```bash
pip install openpyxl
```

### Read Syntax
```python
pd.read_excel(
    io,               # file path or URL
    sheet_name=0,     # sheet name (str) or index (int) — default: first sheet
    header=0,         # row to use as column names
    index_col=None,   # column to use as row index
    usecols=None,     # columns to load
    dtype=None        # force types
)
```

### Write Syntax
```python
df.to_excel(
    excel_writer,        # file path
    sheet_name='Sheet1', # name of the sheet
    index=False          # same as to_csv — almost always False
)
```

> **Note:** Because Excel files require a real file path (not StringIO), we demonstrate the concept with code comments only and note what the output would be.

In [ ]:
# --- 4. read_excel() and to_excel() demo ---
# We simulate what Excel I/O looks like using a temporary file.
# (Requires openpyxl: pip install openpyxl)

import tempfile
import os

df_employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David'],
    'department': ['Engineering', 'Marketing', 'Engineering', 'HR'],
    'salary': [85000, 70000, 91000, 68000],
    'join_year': [2019, 2015, 2021, 2012]
})

# Try to write and read Excel; gracefully handle if openpyxl is missing
try:
    # Write to a temporary Excel file
    with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
        tmp_path = tmp.name

    df_employees.to_excel(tmp_path, sheet_name='Employees', index=False)
    print(f"Written to: {tmp_path}")

    # Read it back — first sheet (default)
    df_from_excel = pd.read_excel(tmp_path, sheet_name='Employees')
    print("\n=== Read back from Excel ===")
    print(df_from_excel)
    print("\nDtypes:")
    print(df_from_excel.dtypes)

    os.remove(tmp_path)  # clean up

except ImportError:
    print("openpyxl not installed. Install it with: pip install openpyxl")
    print("\nExample code (runs when openpyxl is installed):")
    print("  df.to_excel('output.xlsx', sheet_name='Data', index=False)")
    print("  df2 = pd.read_excel('output.xlsx', sheet_name='Data')")
    print("\nDataFrame that would be written:")
    print(df_employees)

---
### 4.1 Reading Multiple Sheets

**`sheet_name` can be:**

| Value | Behaviour |
|-------|-----------|
| `0` (default) | First sheet |
| `'Sheet1'` | Sheet named 'Sheet1' |
| `[0, 1]` | Returns dict of DataFrames for sheets at index 0 and 1 |
| `None` | Returns dict of **all** sheets |

```python
# Read all sheets at once
all_sheets = pd.read_excel('report.xlsx', sheet_name=None)
# all_sheets is a dict: {'Sheet1': df1, 'Sheet2': df2, ...}

for sheet_name, sheet_df in all_sheets.items():
    print(f"Sheet: {sheet_name}, Shape: {sheet_df.shape}")
```

In [ ]:
# --- 4.1 Writing multiple sheets to one Excel file ---

df_engineering = pd.DataFrame({
    'name': ['Alice', 'Carol'],
    'salary': [85000, 91000]
})

df_marketing = pd.DataFrame({
    'name': ['Bob', 'Eve'],
    'salary': [70000, 75000]
})

try:
    import openpyxl
    with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
        tmp_path = tmp.name

    # Use ExcelWriter to write multiple sheets
    with pd.ExcelWriter(tmp_path, engine='openpyxl') as writer:
        df_engineering.to_excel(writer, sheet_name='Engineering', index=False)
        df_marketing.to_excel(writer, sheet_name='Marketing', index=False)

    print(f"Excel file written with 2 sheets.")

    # Read all sheets back
    all_sheets = pd.read_excel(tmp_path, sheet_name=None)
    for name, sdf in all_sheets.items():
        print(f"\nSheet: '{name}'")
        print(sdf)

    os.remove(tmp_path)

except ImportError:
    print("openpyxl not installed. Skipping Excel multi-sheet demo.")
    print("Install with: pip install openpyxl")

---
## 5. `pd.read_json()` and `df.to_json()`

### What is it?
Reads/writes JSON (JavaScript Object Notation) — the standard format for web APIs.

### Read Syntax
```python
pd.read_json(
    path_or_buf,          # file path, URL, or JSON string
    orient=None           # how the JSON is structured (see table below)
)
```

### Write Syntax
```python
df.to_json(
    path_or_buf=None,     # file path or None (returns string)
    orient='records'      # output format
)
```

### The `orient` Parameter — JSON Structure Formats

| `orient` | JSON Shape | Best for |
|----------|-----------|----------|
| `'records'` | `[{col: val, ...}, ...]` | API responses (most common) |
| `'index'` | `{index: {col: val}}` | When row index matters |
| `'columns'` | `{col: {index: val}}` | Pandas default write |
| `'values'` | `[[val, val, ...], ...]` | Raw 2D array |
| `'split'` | `{index: [...], columns: [...], data: [...]}` | Compact with metadata |

In [ ]:
# --- 5.1 read_json() with orient='records' ---
# This is the most common JSON shape from web APIs: a list of objects

json_string = '''[
    {"name": "Alice", "age": 28, "city": "Delhi"},
    {"name": "Bob",   "age": 34, "city": "Mumbai"},
    {"name": "Carol", "age": 26, "city": "Pune"}
]'''

# Read JSON directly from a string
df = pd.read_json(io.StringIO(json_string), orient='records')

print("=== read_json(orient='records') ===")
print(df)
print()
print("Dtypes:")
print(df.dtypes)

In [ ]:
# --- 5.2 to_json() — writing a DataFrame as JSON ---

df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'age': [28, 34, 26],
    'city': ['Delhi', 'Mumbai', 'Pune']
})

# orient='records' — most readable, matches API format
json_records = df.to_json(orient='records', indent=2)
print("=== to_json(orient='records') ===")
print(json_records)
print()

# orient='columns' — Pandas default (less readable)
json_columns = df.to_json(orient='columns', indent=2)
print("=== to_json(orient='columns') — Pandas default ===")
print(json_columns)
print()

# orient='split' — compact with metadata
json_split = df.to_json(orient='split', indent=2)
print("=== to_json(orient='split') ===")
print(json_split)

---
## 6. Practical Tips After Reading Data

After every `read_csv()` / `read_excel()` / `read_json()`, do this **immediately**:

```python
df.head()    # 1. Visual check — does it look right?
df.info()    # 2. Are the dtypes correct? Any non-null counts look off?
df.shape     # 3. Is the number of rows/columns what you expect?
df.isna().sum()  # 4. How much data is missing?
```

### The Read Checklist

| Task | How |
|------|-----|
| Check shape | `df.shape` |
| Check types | `df.dtypes` or `df.info()` |
| Check missing | `df.isna().sum()` |
| Check first rows | `df.head()` |
| Check last rows | `df.tail()` |

In [ ]:
# --- 6. Post-read checklist demo ---

csv_data = """name,age,salary,city,join_date
Alice,28,85000,Delhi,2019-03-15
Bob,34,,Mumbai,2015-07-22
Carol,26,91000,Pune,2021-01-10
David,40,68000,,2012-11-30
Eva,31,95000,Bangalore,2020-05-08
"""

df = pd.read_csv(io.StringIO(csv_data))

print("Step 1: .head() — visual check")
print(df.head())
print()

print("Step 2: .info() — types and non-null counts")
df.info()
print()

print("Step 3: .shape")
print(df.shape)
print()

print("Step 4: .isna().sum() — missing values")
print(df.isna().sum())

---
## 7. Common Mistakes Summary

| ❌ Mistake | ✅ Fix |
|-----------|--------|
| Forgetting `index=False` in `to_csv()` | Always write `df.to_csv('file.csv', index=False)` |
| Not specifying `sep` for `;` or `\t` files | `pd.read_csv('f.csv', sep=';')` |
| Getting wrong dtypes due to missing values | Use `dtype={'col': type}` at load time |
| Missing value strings not treated as NaN | Use `na_values=['--', 'none', '?']` |
| Loading all columns from a large file | Use `usecols=['col1', 'col2']` to save memory |
| Not checking the data after reading | Always run `.head()` and `.info()` right after read |

---
## 🧠 Practice Questions

Try to solve these on your own before looking at the answer key below.

---

**Q1 [Easy]** — Read the following semicolon-separated data into a DataFrame using `io.StringIO`:
```
product;price;quantity
Apple;0.5;100
Banana;0.3;200
Cherry;1.2;50
```

---

**Q2 [Easy]** — From the following CSV, load **only** the `name` and `city` columns:
```
name,age,salary,city,department
Alice,28,85000,Delhi,Engineering
Bob,34,70000,Mumbai,Marketing
Carol,26,91000,Pune,Engineering
```

---

**Q3 [Easy]** — Read this CSV and ensure that `'N/A'`, `'--'`, and `'none'` are all treated as NaN:
```
name,score,grade
Alice,92,A
Bob,--,none
Carol,88,B
David,N/A,C
```

---

**Q4 [Easy]** — Create a DataFrame, write it to a CSV string **without the index**, then read it back and verify the columns do not include `Unnamed: 0`.

---

**Q5 [Medium]** — Read the following CSV and specify `dtype={'salary': float, 'age': int}` at load time. Verify the types are correct:
```
name,age,salary
Alice,28,85000
Bob,34,70000
Carol,26,91000
```

---

**Q6 [Medium]** — Read the following JSON string into a DataFrame using `orient='records'`, then write it back to JSON with `orient='split'`:
```json
[{"city": "Delhi", "population": 32941000}, {"city": "Mumbai", "population": 20667656}, {"city": "Bangalore", "population": 13193000}]
```

---

**Q7 [Medium]** — From the CSV below:  
1. Read it in  
2. Drop rows with any NaN  
3. Convert `age` to int  
4. Write **only** the `name` and `salary` columns to a new CSV string (no index)
```
name,age,salary
Alice,28,85000
Bob,,70000
Carol,26,91000
David,40,
Eva,31,95000
```

---
## ✅ Answer Key

In [ ]:
# =====================================================================
# Q1 [Easy]: Read semicolon-separated CSV using StringIO
# =====================================================================

csv_q1 = """product;price;quantity
Apple;0.5;100
Banana;0.3;200
Cherry;1.2;50
"""

df_q1 = pd.read_csv(io.StringIO(csv_q1), sep=';')

print("Q1 Answer:")
print(df_q1)
print(f"\nColumns: {df_q1.columns.tolist()}")
print(f"Dtypes:\n{df_q1.dtypes}")

In [ ]:
# =====================================================================
# Q2 [Easy]: Load only 'name' and 'city' columns using usecols
# =====================================================================

csv_q2 = """name,age,salary,city,department
Alice,28,85000,Delhi,Engineering
Bob,34,70000,Mumbai,Marketing
Carol,26,91000,Pune,Engineering
"""

df_q2 = pd.read_csv(io.StringIO(csv_q2), usecols=['name', 'city'])

print("Q2 Answer:")
print(df_q2)
print(f"\nColumns loaded: {df_q2.columns.tolist()}")
print(f"Shape: {df_q2.shape}  (5 columns in file, only 2 loaded)")

In [ ]:
# =====================================================================
# Q3 [Easy]: Treat 'N/A', '--', 'none' as NaN using na_values
# =====================================================================

csv_q3 = """name,score,grade
Alice,92,A
Bob,--,none
Carol,88,B
David,N/A,C
"""

df_q3 = pd.read_csv(
    io.StringIO(csv_q3),
    na_values=['--', 'none', 'N/A']
)

print("Q3 Answer:")
print(df_q3)
print()
print("NaN count per column:")
print(df_q3.isna().sum())
# score: Bob=NaN, David=NaN → 2
# grade: Bob=NaN → 1

In [ ]:
# =====================================================================
# Q4 [Easy]: Write with index=False, read back, verify no 'Unnamed: 0'
# =====================================================================

df_q4 = pd.DataFrame({
    'product': ['Pen', 'Notebook', 'Stapler'],
    'price': [10, 45, 85],
    'stock': [200, 80, 30]
})

# Write to CSV string (no index)
csv_string = df_q4.to_csv(index=False)
print("CSV string written:")
print(csv_string)

# Read it back
df_q4_back = pd.read_csv(io.StringIO(csv_string))
print("Read back:")
print(df_q4_back)
print()

# Verify no unwanted index column
has_unnamed = any('Unnamed' in col for col in df_q4_back.columns)
print(f"Columns: {df_q4_back.columns.tolist()}")
print(f"Has 'Unnamed' column? {has_unnamed}")

In [ ]:
# =====================================================================
# Q5 [Medium]: Specify dtype={'salary': float, 'age': int} at load time
# =====================================================================

csv_q5 = """name,age,salary
Alice,28,85000
Bob,34,70000
Carol,26,91000
"""

# Without dtype
df_auto = pd.read_csv(io.StringIO(csv_q5))
print("=== Auto-detected dtypes ===")
print(df_auto.dtypes)
print()

# With dtype specified
df_q5 = pd.read_csv(
    io.StringIO(csv_q5),
    dtype={'salary': float, 'age': int}
)

print("=== With dtype={'salary': float, 'age': int} ===")
print(df_q5)
print()
print("Dtypes:")
print(df_q5.dtypes)
print()

# Verify
assert df_q5['age'].dtype == 'int64', "age should be int64"
assert df_q5['salary'].dtype == 'float64', "salary should be float64"
print("✅ dtype verification passed!")

In [ ]:
# =====================================================================
# Q6 [Medium]: Read JSON with orient='records', write with orient='split'
# =====================================================================

json_q6 = '[{"city": "Delhi", "population": 32941000}, {"city": "Mumbai", "population": 20667656}, {"city": "Bangalore", "population": 13193000}]'

# Read with orient='records'
df_q6 = pd.read_json(io.StringIO(json_q6), orient='records')
print("=== Read with orient='records' ===")
print(df_q6)
print(f"\nDtypes:\n{df_q6.dtypes}")
print()

# Write back with orient='split'
json_split_output = df_q6.to_json(orient='split', indent=2)
print("=== Written with orient='split' ===")
print(json_split_output)
print()

# Read the split-oriented JSON back to verify round-trip
df_q6_back = pd.read_json(io.StringIO(json_split_output), orient='split')
print("=== Round-trip verify ===")
print(df_q6_back)

In [ ]:
# =====================================================================
# Q7 [Medium]: Read → drop NaN → fix types → write specific columns
# =====================================================================

csv_q7 = """name,age,salary
Alice,28,85000
Bob,,70000
Carol,26,91000
David,40,
Eva,31,95000
"""

# Step 1: Read
df_q7 = pd.read_csv(io.StringIO(csv_q7))
print("Step 1: Read")
print(df_q7)
print(f"Shape: {df_q7.shape}")
print()

# Step 2: Drop rows with any NaN
df_q7_clean = df_q7.dropna()
print("Step 2: After dropna()")
print(df_q7_clean)
print(f"Shape: {df_q7_clean.shape}")
print()

# Step 3: Convert age to int (was float64 after reading because of NaN)
df_q7_clean = df_q7_clean.copy()  # avoid SettingWithCopyWarning
df_q7_clean['age'] = df_q7_clean['age'].astype(int)
print("Step 3: After astype(int) on age")
print(df_q7_clean.dtypes)
print()

# Step 4: Write only 'name' and 'salary' columns, no index
final_csv = df_q7_clean.to_csv(index=False, columns=['name', 'salary'])
print("Step 4: Final CSV (name + salary only)")
print(final_csv)

# Verify the output
df_verify = pd.read_csv(io.StringIO(final_csv))
print("Verification — read back:")
print(df_verify)
print(f"Columns: {df_verify.columns.tolist()}")

---
## 📋 Quick Revision Table

| Function | Key Parameters | Common Pitfall |
|----------|---------------|----------------|
| `pd.read_csv()` | `sep`, `header`, `index_col`, `usecols`, `dtype`, `nrows`, `skiprows`, `na_values` | Forgetting `sep` for non-comma files |
| `df.to_csv()` | `index=False`, `sep`, `columns`, `na_rep` | Default `index=True` creates extra column |
| `pd.read_excel()` | `sheet_name`, `header`, `usecols`, `dtype` | Requires `openpyxl` installed |
| `df.to_excel()` | `sheet_name`, `index=False` | Requires `openpyxl` installed |
| `pd.read_json()` | `orient` | Default orient may not match your JSON structure |
| `df.to_json()` | `orient`, `indent` | Default `orient='columns'` is rarely what APIs expect |
| `io.StringIO()` | N/A | Wrap a string to simulate a file object for reading |

---

### Parameter Quick Reference

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `sep=';'` | Non-comma delimiter | `sep='\t'` for TSV |
| `header=None` | No header row in file | + `names=[...]` to name columns |
| `index_col='id'` | Column to use as index | Avoids separate `.set_index()` |
| `usecols=['a','b']` | Load only these columns | Big memory saving on wide files |
| `dtype={'age': int}` | Force types at load | Avoids object dtype surprise |
| `nrows=1000` | Load first N rows | Great for previewing large files |
| `skiprows=2` | Skip top rows | For files with metadata headers |
| `na_values=['--']` | Extra NaN strings | Catches non-standard sentinels |
| `index=False` | Don't write row index | Almost always needed in `to_csv()` |
| `na_rep='N/A'` | NaN placeholder in output | For downstream systems |
| `orient='records'` | JSON as list of dicts | Most readable / API-friendly |

---
## 🎤 3 Interview Questions

---

**Q1:** You have a large CSV file with 100 columns and 10 million rows, but you only need 5 columns for your analysis. What two things would you do to reduce memory usage when reading it?

> **Answer:** Use `usecols=['col1', 'col2', ...]` to load only the required columns, and `dtype={...}` to specify the most memory-efficient types (e.g., `int32` instead of `int64`, or `category` for low-cardinality strings).

---

**Q2:** A colleague gives you a CSV file. After reading it with `pd.read_csv()` and then re-writing with `df.to_csv('out.csv')` and reading it again, you notice an `Unnamed: 0` column. What caused this and how do you fix it?

> **Answer:** `to_csv()` defaults to `index=True`, which writes the integer row index as an unlabelled first column. When read back, Pandas names it `Unnamed: 0`. Fix: always use `df.to_csv('out.csv', index=False)`.

---

**Q3:** What is `io.StringIO` and why is it useful when working with Pandas I/O functions?

> **Answer:** `io.StringIO` is a Python built-in that wraps a plain string and makes it behave like a file object. It lets you pass CSV/JSON content as an in-memory string to functions like `pd.read_csv()` and `pd.read_json()` — which normally expect a file path — without needing any actual file on disk. This is especially useful for testing, unit tests, and notebooks where you want self-contained reproducible examples.

---
## ➡️ What's Next

**Notebook 12 — Performance & Best Practices**

Now that you know how to read and write data, the next notebook focuses on doing it **efficiently at scale**:

- **Vectorization** — why loops are slow and how to avoid them
- **`apply()` vs vectorized operations** — when to use which
- **Memory optimization** — `category` dtype, downcasting, chunked reading
- **`pd.eval()` and `pd.query()`** — faster filtering for large DataFrames
- **Profiling** — finding bottlenecks with `%timeit` and `memory_profiler`
- **Chunked reading with `chunksize=`** — processing files larger than RAM

---

*End of Notebook 11 — Reading & Writing Data*  
*Pandas from First Principles Series*